# 🐙 Day-2 · Notebook 04: Git, GitHub & Professional Collaboration
**Python for AI Developers · 20-Minute Professional Developer Mastery**

Welcome to the ultimate Git & GitHub guide for AI engineers!

In this notebook, we master every command you will use daily in software and AI engineering:
1. **Why do we use Git?**: Distributed version control, time-travel history, and collaboration.
2. **The 4 Git Zones**: Working Directory, Staging Area, Local Repo, Remote GitHub (with visual highway!).
3. **The Daily Command Cycle**: `git status`, `git add`, `git commit -m`, `git push`.
4. **`git checkout` vs Modern `git switch` & `git restore`**.
5. **Branching & Merging**: Creating feature branches and resolving merge conflicts cleanly.
6. **`git stash` & `git stash pop`**: Pausing your work without making messy commits.
7. **Git Hooks (`pre-commit`)**: Automatically blocking leaked `.env` API keys and multi-GB model weights!
8. **Pull Requests (PRs)**: Professional code review, CI/CD checks, and branch protection.

Let's dive in!


### 🛣️ The Git Collaboration Highway Visualized

![Git Collaboration Highway](../../images/day2_git_workflow_diagram.png)

```
[Working Directory]  --( git add . )-->  [Staging Area]  --( git commit )-->  [Local Repo]  --( git push )-->  [Remote GitHub]
 (Active edits)                           (Packed box)                         (Milestone hash)                   (Cloud sync)
```


In [ ]:
# Inspecting Git in Python:
import subprocess

def run_git(cmd: str):
    """Runs git commands cleanly and displays output."""
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"$ {cmd}")
    if res.stdout.strip():
        print(res.stdout.strip())
    if res.stderr.strip() and res.returncode != 0:
        print("Stderr:", res.stderr.strip())

run_git("git --version")


---
### 🔀 Concept 1: `git checkout` vs Modern `git switch` & `git restore`

In older Git versions, `git checkout` was overloaded: it was used to switch branches AND to discard uncommitted file changes. This caused accidental data loss!

Git introduced two clear, dedicated commands:

| Task | Modern Preferred Command | Legacy Command |
| :--- | :--- | :--- |
| **Create & switch to new branch** | `git switch -c feature/rag-pipeline` | `git checkout -b feature/rag-pipeline` |
| **Switch to existing branch** | `git switch main` | `git checkout main` |
| **Discard uncommitted file changes** | `git restore model_training.py` | `git checkout -- model_training.py` |
| **Unstage a file from staging area** | `git restore --staged model.py` | `git reset HEAD model.py` |

> **Rule of Thumb**: Use `git switch` for branches, and `git restore` for files!

---
### 📦 Concept 2: `git stash` (The Developer's Pause Button)

Imagine you are halfway through writing an AI prompt engine, and your professor or manager suddenly says: *'Hotfix needed on main immediately!'*

You can't commit incomplete broken code, but you can't switch branches with dirty edits either.
**Solution: `git stash`!**

```bash
# 1. Shelve all uncommitted work into Git's temporary memory pocket:
git stash save "WIP: prompt engine halfway"

# 2. Your working directory is now 100% clean! Switch branch and fix bug:
git switch main
# ... fix bug & push ...

# 3. Return to your feature branch and restore your shelved work:
git switch feature/rag-pipeline
git stash pop
```


---
### ⚔️ Concept 3: `git merge` & Resolving Merge Conflicts

When two teammates edit the same line of code on different branches, Git pauses the merge and inserts **conflict markers**:

```python
<<<<<<< HEAD (Current branch: main)
STUDENT_API_URL = "https://api.kit.edu/v1/students"
=======
STUDENT_API_URL = "https://api.kit.edu/v2/ai-students"
>>>>>>> feature/ai-upgrade (Incoming branch)
```

**How to Resolve a Conflict in 3 Steps:**
1. Open the file in VS Code.
2. Decide which code to keep (or combine both). Delete `<<<<<<<`, `=======`, and `>>>>>>>` markers!
3. Stage and commit: `git add . && git commit -m "fix: resolve merge conflict on student API URL"`.


---
### 🪝 Concept 4: Git Hooks (`pre-commit`) for AI Developers

**What is a Git Hook?**
A script located in `.git/hooks/` that triggers automatically when an event happens (like before committing or before pushing).

**Why are hooks critical in AI?**
1. **Never leak API keys**: Scan code before commit to block accidental pushes of `OPENAI_API_KEY` or `GEMINI_API_KEY`.
2. **Block multi-gigabyte models**: Block `.pt` PyTorch weights or `.parquet` datasets (>100MB) from bloating Git repos.
3. **Code Quality**: Run `black` and `flake8` auto-formatters before every commit.

Let's see what a production AI `.pre-commit-config.yaml` looks like:

In [ ]:
pre_commit_config = """# .pre-commit-config.yaml for AI Teams
repos:
  - repo: https://github.com/pre-commit/pre-commit-hooks
    rev: v4.4.0
    hooks:
      - id: check-added-large-files
        args: ['--maxkb=50000']  # Blocks files > 50MB (model weights!)
      - id: detect-private-key    # Catches AWS, SSH, and OpenAI keys
      - id: check-yaml
      - id: end-of-file-fixer

  - repo: https://github.com/psf/black
    rev: 23.3.0
    hooks:
      - id: black                # Formats Python code automatically
"""

print("🛡️ Production AI Pre-Commit Hook Configuration:")
print(pre_commit_config)
print("Install in your terminal via: pip install pre-commit && pre-commit install")


---
### 📋 Step-by-Step Production Guide: Initializing & Pushing to GitHub

Here are the exact commands with production commentary to initialize this project locally and push to GitHub:

```bash
# =============================================================================
# STEP 1: INITIALIZE LOCAL GIT REPOSITORY
# =============================================================================
# Run inside the project root folder (python_ai_developer/):
git init

# Set your committer name & email (appears in GitHub commit history):
git config user.name "Your Name"
git config user.email "your.email@example.com"

# =============================================================================
# STEP 2: CREATE / VERIFY .GITIGNORE
# =============================================================================
# Ensure .gitignore blocks API keys (.env), __pycache__, and gigabyte weights (.safetensors, .pt)
# A full production template is available in GIT_AND_GITHUB_GUIDE.md!

# =============================================================================
# STEP 3: STAGE SPECIFIC CURRICULUM ASSETS (AVOID BLIND 'git add .')
# =============================================================================
# Stage Day 2 & Day 3 notebooks:
git add notebooks/day_02/ notebooks/day_03/

# Stage presentations and diagrams:
git add presentations/ images/

# Stage hands-on labs and notes:
git add hands_on_labs/ notes/

# Stage guides and documentation:
git add GIT_AND_GITHUB_GUIDE.md README.md .gitignore

# Verify staged files:
git status

# =============================================================================
# STEP 4: CREATE CONVENTIONAL COMMIT
# =============================================================================
git commit -m "feat(curriculum): complete Day 2 OOP & Day 3 Generative AI master modules with notebooks, slides, and guides"

# =============================================================================
# STEP 5: RENAME BRANCH & LINK TO GITHUB REMOTE
# =============================================================================
# Rename active branch to modern standard 'main':
git branch -M main

# Link your remote GitHub repository (replace with your actual GitHub repo URL):
git remote add origin https://github.com/<YOUR_USERNAME>/<YOUR_REPO_NAME>.git

# Verify remote URL:
git remote -v

# =============================================================================
# STEP 6: PUSH CODE TO GITHUB
# =============================================================================
# Push main branch and set upstream tracking (-u):
git push -u origin main
```

> 📖 For full branching workflows, merge conflict resolution, and PR guidelines, see [GIT_AND_GITHUB_GUIDE.md](../../GIT_AND_GITHUB_GUIDE.md)!

---
### 🚀 Concept 5: Pull Requests (PRs) & Team Code Reviews

In production engineering, you **never** push directly to `main`.
Instead, you follow the **Pull Request (PR) Highway**:

```bash
# 1. Create feature branch:
git switch -c feat/kit-student-api

# 2. Code, test, and commit:
git add .
git commit -m "feat(kit): implement student USN database integration"

# 3. Push branch to GitHub:
git push -u origin feat/kit-student-api
```

4. Go to GitHub and click **'Open Pull Request'**.
5. Teammates review your diffs, leave comments, and automated CI pipelines run tests (`pytest`).
6. Once approved, click **'Squash and Merge'** to deploy your code into production!

---
### 🎓 Day 2 Complete: You are a Professional Python & AI Developer!

You have mastered:
- Functions, `*args/**kwargs`, and `@track_execution_time` decorators.
- Full KIT College OOP: USN standard, Abstraction, Encapsulation, 5 Inheritance Types, MRO C3 Linearization, Polymorphism, and Operator Overloading.
- Defensive engineering: 4-stage exception lifecycle and `with open()` file streams.
- Git & GitHub: 4 zones, `git switch`, `git stash`, conflict resolution, pre-commit hooks, and PRs.

Next stop: **Day 3 · Generative AI & Prompt Engineering Masterclass**! 🚀
